In [4]:
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(project_root)

c:\Users\Макетолог\Документы\MYPROJECT\data-lab


In [2]:
from datetime import date, timedelta

from src.common.ch_client import ch_select, ch_execute
from src.pipelines.cbr_pipeline import get_cbr_rates_history
from src.load.clickhouse_load import (
    delete_currency_rates_by_source_date,
    load_currency_rates_to_clickhouse
)

In [3]:
import importlib
import src.load.clickhouse_load

importlib.reload(src.load.clickhouse_load)

from src.load.clickhouse_load import load_currency_rates_to_clickhouse

In [4]:
from datetime import date, timedelta

end_date = date.today()
start_date = end_date - timedelta(weeks=1)


In [72]:
delete_result = delete_currency_rates_by_source_date(
    start_date,
    end_date
)

delete_result

{'table_name': 'data_lab.raw_cbr_rates',
 'start_date': datetime.date(2026, 4, 18),
 'end_date': datetime.date(2026, 7, 17),
 'status': 'success'}

In [47]:
load_result = load_currency_rates_to_clickhouse(
    currency_rates_history_df
)

load_result

{'table_name': 'data_lab.raw_cbr_rates',
 'rows_loaded': 2052,
 'status': 'success'}

In [54]:
row_count_df = ch_select("""
    SELECT
        count(*) AS row_count
    FROM data_lab.raw_cbr_rates
""")

row_count_df

,row_count
0,7398


In [46]:
duplicate_check_df = ch_select("""
    SELECT
        source_date,
        currency_code,
        count(*) AS row_count
    FROM data_lab.raw_cbr_rates
    GROUP BY
        source_date,
        currency_code
    HAVING row_count > 1
    ORDER BY
        source_date,
        currency_code
""")

duplicate_check_df

""


In [18]:
from src.quality.cbr_checks import validate_cbr_rates_df

In [53]:
validation_result = validate_cbr_rates_df(
    currency_rates_history_df
)

validation_result

{'status': 'success', 'row_count': 2052, 'errors': []}

In [52]:
load_result = load_currency_rates_to_clickhouse(
    currency_rates_history_df
)

load_result

{'table_name': 'data_lab.raw_cbr_rates',
 'rows_loaded': 2052,
 'status': 'success'}

In [51]:
broken_currency_rates_df = currency_rates_history_df.copy()

broken_currency_rates_df.loc[
    0,
    'rate'
] = -1

In [22]:
load_currency_rates_to_clickhouse(
    broken_currency_rates_df
)

{'table_name': 'data_lab.raw_cbr_rates',
 'rows_loaded': 0,
 'status': 'failed',
 'validation_errors': ['Invalid rate values: 1']}

In [12]:
from datetime import date, timedelta
from src.pipelines.cbr_pipeline import run_cbr_rates_pipeline

end_date=date.today()
start_date=end_date - timedelta(days=30)

pipeline_result = run_cbr_rates_pipeline(
    start_date=start_date,
    end_date=end_date
)

pipeline_result

{'pipeline_name': 'cbr_rates_pipeline',
 'start_date': datetime.date(2026, 6, 17),
 'end_date': datetime.date(2026, 7, 17),
 'rows_loaded': 124,
 'status': 'success',
 'delete_result': {'table_name': 'data_lab.raw_cbr_rates',
  'start_date': datetime.date(2026, 6, 17),
  'end_date': datetime.date(2026, 7, 17),
  'status': 'success'},
 'load_result': {'table_name': 'data_lab.raw_cbr_rates',
  'rows_loaded': 124,
  'status': 'success'}}

In [10]:
currency_rates_history_df = get_cbr_rates_history(
    start_date=start_date,
    end_date=end_date
)

currency_rates_history_df['currency_code'].unique()

array(['USD', 'EUR', 'CNY', 'JPY'], dtype=object)

In [11]:
from src.pipelines.cbr_pipeline import run_cbr_rates_pipeline

pipeline_result = run_cbr_rates_pipeline(
    start_date=start_date,
    end_date=end_date
)

pipeline_result['validation_result']

{'status': 'success', 'row_count': 28, 'errors': []}

In [22]:
from src.pipelines.cbr_pipeline import get_date_range

request_dates = get_date_range(
    start_date,
    end_date
)

request_dates

[datetime.date(2026, 1, 30),
 datetime.date(2026, 1, 31),
 datetime.date(2026, 2, 1),
 datetime.date(2026, 2, 2),
 datetime.date(2026, 2, 3),
 datetime.date(2026, 2, 4),
 datetime.date(2026, 2, 5),
 datetime.date(2026, 2, 6),
 datetime.date(2026, 2, 7),
 datetime.date(2026, 2, 8),
 datetime.date(2026, 2, 9),
 datetime.date(2026, 2, 10),
 datetime.date(2026, 2, 11),
 datetime.date(2026, 2, 12),
 datetime.date(2026, 2, 13),
 datetime.date(2026, 2, 14),
 datetime.date(2026, 2, 15),
 datetime.date(2026, 2, 16),
 datetime.date(2026, 2, 17),
 datetime.date(2026, 2, 18),
 datetime.date(2026, 2, 19),
 datetime.date(2026, 2, 20),
 datetime.date(2026, 2, 21),
 datetime.date(2026, 2, 22),
 datetime.date(2026, 2, 23),
 datetime.date(2026, 2, 24),
 datetime.date(2026, 2, 25),
 datetime.date(2026, 2, 26),
 datetime.date(2026, 2, 27),
 datetime.date(2026, 2, 28),
 datetime.date(2026, 3, 1),
 datetime.date(2026, 3, 2),
 datetime.date(2026, 3, 3),
 datetime.date(2026, 3, 4),
 datetime.date(2026, 3, 5),

In [23]:
import pandas as pd

request_dates_df = pd.DataFrame({
    'request_dates': request_dates
})

request_dates_df

,request_dates
0,2026-01-30
1,2026-01-31
2,2026-02-01
3,2026-02-02
4,2026-02-03
...,...
164,2026-07-13
165,2026-07-14
166,2026-07-15
167,2026-07-16


In [7]:
from src.pipelines.cbr_pipeline import run_cbr_rates_pipeline

pipeline_result = run_cbr_rates_pipeline(
    start_date=start_date,
    end_date=end_date
)

pipeline_result

{'pipeline_name': 'cbr_rates_pipeline',
 'start_date': datetime.date(2026, 1, 30),
 'end_date': datetime.date(2026, 7, 17),
 'rows_loaded': 676,
 'status': 'success',
 'validation_result': {'status': 'success', 'row_count': 676, 'errors': []},
 'delete_result': {'table_name': 'data_lab.raw_cbr_rates',
  'start_date': datetime.date(2026, 1, 30),
  'end_date': datetime.date(2026, 7, 17),
  'status': 'success'},
 'load_result': {'table_name': 'data_lab.raw_cbr_rates',
  'rows_loaded': 676,
  'status': 'success'}}

In [27]:
row_count_df = ch_select(f"""
    SELECT
        count() as row_count
        FROM data_lab.raw_cbr_rates
    WHERE source_date BETWEEN '{start_date}' and '{end_date}'
""")

row_count_df

,row_count
0,676


In [32]:
duplicates_df = ch_select(f"""
                        SELECT 
                            source_date,
                            currency_code,
                            count() AS row_count
                        FROM data_lab.raw_cbr_rates
                        WHERE source_date BETWEEN '{start_date}' and '{end_date}'
                        GROUP BY 
                            source_date,
                            currency_code
                        HAVING count() > 1
                        """)
duplicates_df

""


In [16]:
from datetime import datetime, timezone
import pandas as pd

from src.pipelines.cbr_pipeline import get_cbr_rates_history

print(datetime.now())
print(pd.Timestamp.now())
print(datetime.now(timezone.utc))

currency_rates_history_df = get_cbr_rates_history(
    start_date=start_date,
    end_date=end_date
)
print(currency_rates_history_df['load_dttm'].head())

2026-07-17 15:55:09.225800
2026-07-17 15:55:09.226803
2026-07-17 05:55:09.226803+00:00
0   2026-07-17 15:55:10.246085
1   2026-07-17 15:55:10.246085
2   2026-07-17 15:55:10.246085
3   2026-07-17 15:55:10.246085
4   2026-07-17 15:55:11.031006
Name: load_dttm, dtype: datetime64[us]


In [3]:
from src.config.settings import (CBR_RATES_STAGING_TABLE_NAME)

from src.load.clickhouse_load import get_table_row_count

row_count = get_table_row_count(
    CBR_RATES_STAGING_TABLE_NAME
)

row_count

0

In [7]:
from src.load.clickhouse_load import (
    load_currency_rates_to_clickhouse,
)

from src.extract.cbr_api import (
    get_cbr_rates_by_date,
)

currency_rates_df = get_cbr_rates_by_date(
    request_date=end_date,
)

load_result = load_currency_rates_to_clickhouse(
    currency_rates_df=currency_rates_df,
    table_name=CBR_RATES_STAGING_TABLE_NAME
)

In [2]:
from src.pipelines.cbr_pipeline import run_cbr_rates_pipeline

from datetime import date, timedelta

end_date = date.today()
start_date = end_date - timedelta(weeks=1)

In [3]:
result = run_cbr_rates_pipeline(
    start_date=start_date,
    end_date=end_date
)

result

2026-07-22 15:38:37 | INFO | src.pipelines.cbr_pipeline | Запуск pipeline ЦБ за период 2026-07-15 - 2026-07-22
2026-07-22 15:38:44 | INFO | src.pipelines.cbr_pipeline | Получено строк после extract и transform: 32
2026-07-22 15:38:44 | INFO | src.pipelines.cbr_pipeline | Проверка качества пройдена. Количество строк: 32
2026-07-22 15:38:44 | INFO | src.pipelines.cbr_pipeline | Staging таблица очищена: data_lab.raw_cbr_rates_staging
2026-07-22 15:38:44 | INFO | src.pipelines.cbr_pipeline | Загрузка в staging завершена. Загружено строк: 32
2026-07-22 15:38:45 | INFO | src.pipelines.cbr_pipeline | Проверка staging пройдена.
2026-07-22 15:38:45 | INFO | src.pipelines.cbr_pipeline | Удаление старых данных из raw таблицы завершено со статусом: success
2026-07-22 15:38:45 | INFO | src.pipelines.cbr_pipeline | Публикация staging в raw завершена. Загружено строк 32
2026-07-22 15:38:45 | INFO | src.pipelines.cbr_pipeline | Pipeline ЦБ успешно завершен. Загружено строк в raw-таблицу: 32


{'pipeline_name': 'cbr_rates_pipeline',
 'start_date': datetime.date(2026, 7, 15),
 'end_date': datetime.date(2026, 7, 22),
 'rows_extracted': 32,
 'rows_loaded': 32,
 'status': 'success',
 'stage': 'completed',
 'validation_result': {'status': 'success', 'row_count': 32, 'errors': []},
 'staging_load_result': {'table_name': 'data_lab.raw_cbr_rates_staging',
  'rows_loaded': 32,
  'status': 'success'},
 'delete_result': {'table_name': 'data_lab.raw_cbr_rates',
  'start_date': datetime.date(2026, 7, 15),
  'end_date': datetime.date(2026, 7, 22),
  'status': 'success'},
 'load_result': {'status': 'success',
  'rows_loaded': 32,
  'staging_table_name': 'data_lab.raw_cbr_rates_staging',
  'raw_table_name': 'data_lab.raw_cbr_rates'},
 'error': None}